# Prepare Table Egg Price Data

Retrieve and prepare national table egg price data from USDA NASS Quick Stats API. This notebook obtains price received data, cleans the values, and checks for data completeness across years.

In [ ]:
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import sys 
from pathlib import Path

# Load NASS API key from environment
src_path = Path.cwd() / 'src'
sys.path.insert(0, str(src_path))
from utils.nass_api import NASS_API_KEY

## Load Libraries and API Key

In [ ]:
# Query NASS API for national egg prices
base_url = "https://quickstats.nass.usda.gov/api/api_GET/"

params = {'key': NASS_API_KEY,
          'source_desc': 'SURVEY',
          'sector_desc': 'ANIMALS & PRODUCTS',
          'group_desc': 'POULTRY',
          'commodity_desc': 'EGGS',
          'statisticcat_desc': 'PRICE RECEIVED',
          'agg_level_desc': 'NATIONAL'
         }

response = requests.get(base_url, params=params)
egg_prices = response.json()
egg_prices = pd.DataFrame(egg_prices['data'])

## Retrieve Egg Price Data

## Clean and Prepare Data

Select relevant columns for analysis.

In [ ]:
select_vars = ['year', 'reference_period_desc', 'begin_code', 'freq_desc',  'unit_desc', 'sector_desc',  'class_desc', 'commodity_desc', 'short_desc', 'statisticcat_desc', 'Value']

egg_prices = egg_prices[select_vars]

Remove suppressed values (D) and withheld values (Z), then convert to numeric type.

In [ ]:
# Remove rows with suppressed (D) or withheld (Z) values
value_clean = egg_prices['Value'].fillna('').astype(str)
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
egg_prices = egg_prices[~mask].copy()

# Clean and convert Value column to numeric
egg_prices['Value'] = (
    egg_prices['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
egg_prices['Value'] = pd.to_numeric(egg_prices['Value'], errors='coerce')

# Check for missing values
print(f"Missing values in Value column: {egg_prices['Value'].isna().sum()}")

## Data Quality Checks

Check for missing years in the time series.

In [ ]:
# Get years present in the data
actual_years = set(egg_prices['year'].astype(int).unique())

# Create full range of expected years
expected_years = set(range(min(actual_years), max(actual_years) + 1))

# Find missing years
missing_years = expected_years - actual_years

print(f"Missing years: {sorted(missing_years)}")
print(f"Total missing years: {len(missing_years)}")
print(f"Years available: {min(actual_years)} to {max(actual_years)}")
print(f"Total years available: {len(actual_years)}")

Ensure year column is integer type for downstream analysis.

In [ ]:
egg_prices['year'] = egg_prices['year'].astype(int)

## Save Cleaned Data

In [ ]:
egg_prices.to_csv('data/egg_table_prices_nass.csv', index=False)